In [8]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1"
)

cutout_dir = project_dir / "data" / "cutouts_099"
catalog_file = project_dir / "data" / "representative_dmdgs.csv"

subhalo_ids = [
    738558,
    355308,
    617680,
    1436856,
    340851,
]
BOX_SIZE = 205000.0 

catalog = pd.read_csv(catalog_file)
catalog["SubhaloID"] = catalog["SubhaloID"].astype(int)

for subhalo_id in subhalo_ids:

    print("=" * 70)
    print(f"SUBHALO {subhalo_id}")
    print("=" * 70)

    row = catalog[catalog["SubhaloID"] == subhalo_id].iloc[0]
    center = np.array([
        row["x"],
        row["y"],
        row["z"]
    ])

    Rh = row["R_half_star"]

    print(f"Galaxy center: {center}")
    print(f"Stellar half-mass radius: {Rh:.4f} ckpc/h")
    print()

    filename = cutout_dir / f"subhalo_{subhalo_id}_cutout.hdf5"

    with h5py.File(filename, "r") as f:

        def get_radii(coords):
            delta = coords - center
            delta = delta - BOX_SIZE * np.round(delta / BOX_SIZE)

            return np.sqrt(np.sum(delta**2, axis=1))


        if "PartType1" in f:

            dm_coords = f["PartType1"]["Coordinates"][:]

            dm_r = get_radii(dm_coords)

            print(f"DM particles:     {len(dm_r)}")
            print(f"DM r min:         {dm_r.min():.4f}")
            print(f"DM r max:         {dm_r.max():.4f}")

        else:

            dm_r = np.array([])

            print("DM particles:     0")

        if "PartType4" in f:

            star_coords = f["PartType4"]["Coordinates"][:]

            star_r = get_radii(star_coords)

            print(f"Star particles:   {len(star_r)}")
            print(f"Star r min:       {star_r.min():.4f}")
            print(f"Star r max:       {star_r.max():.4f}")

        else:

            star_r = np.array([])

            print("Star particles:   0")

        if "PartType0" in f:

            gas_coords = f["PartType0"]["Coordinates"][:]

            gas_r = get_radii(gas_coords)

            print(f"Gas cells:        {len(gas_r)}")
            print(f"Gas r min:        {gas_r.min():.4f}")
            print(f"Gas r max:        {gas_r.max():.4f}")

        else:

            gas_r = np.array([])

            print("Gas cells:        0")
    print()
    print(f"2 × R_half = {2 * Rh:.4f} ckpc/h")

    dm_inside = np.sum(dm_r <= 2 * Rh)
    star_inside = np.sum(star_r <= 2 * Rh)
    gas_inside = np.sum(gas_r <= 2 * Rh)

    print()
    print("Particles/cells inside 2 R_half:")
    print(f"  DM:    {dm_inside}")
    print(f"  stars: {star_inside}")
    print(f"  gas:   {gas_inside}")

    print()

SUBHALO 738558
Galaxy center: [ 45298.566 164908.9    49094.62 ]
Stellar half-mass radius: 0.6847 ckpc/h

DM particles:     28
DM r min:         0.1108
DM r max:         1.6862
Star particles:   332
Star r min:       0.0103
Star r max:       1.5676
Gas cells:        0

2 × R_half = 1.3694 ckpc/h

Particles/cells inside 2 R_half:
  DM:    26
  stars: 318
  gas:   0

SUBHALO 355308
Galaxy center: [157154.16  128159.61   13816.756]
Stellar half-mass radius: 1.5633 ckpc/h

DM particles:     27
DM r min:         0.0047
DM r max:         4.0643
Star particles:   340
Star r min:       0.0899
Star r max:       4.9524
Gas cells:        0

2 × R_half = 3.1266 ckpc/h

Particles/cells inside 2 R_half:
  DM:    23
  stars: 308
  gas:   0

SUBHALO 617680
Galaxy center: [62048.203 61427.664 41123.05 ]
Stellar half-mass radius: 0.7739 ckpc/h

DM particles:     33
DM r min:         0.3485
DM r max:         2.2587
Star particles:   357
Star r min:       0.0018
Star r max:       2.6271
Gas cells:        

In [12]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
DM_PARTICLE_MASS = 0.00398342749867548
h = 0.6774
profile_dir = project_dir / "data" / "particle_profiles_099"
profile_dir.mkdir(parents=True, exist_ok=True)
all_profiles = []
for subhalo_id in subhalo_ids:

    print()
    print("=" * 70)
    print(f"MASS PROFILE: SUBHALO {subhalo_id}")
    print("=" * 70)

    row = catalog[catalog["SubhaloID"] == subhalo_id].iloc[0]

    center = np.array([
        row["x"],
        row["y"],
        row["z"]
    ])

    Rh = row["R_half_star"]
    catalog_dm_2Rh = row["M_DM_2Rh"]
    catalog_total_2Rh = row["M_total_2Rh"]

    filename = cutout_dir / f"subhalo_{subhalo_id}_cutout.hdf5"

    with h5py.File(filename, "r") as f:

        def get_radii(coords):

            delta = coords - center
            delta = delta - BOX_SIZE * np.round(delta / BOX_SIZE)

            return np.sqrt(np.sum(delta**2, axis=1))

        if "PartType1" in f:

            dm_coords = f["PartType1"]["Coordinates"][:]

            dm_r = get_radii(dm_coords)
            dm_mass = np.full(
                len(dm_r),
                DM_PARTICLE_MASS
            )

        else:

            dm_r = np.array([])
            dm_mass = np.array([])

        if "PartType4" in f:

            star_coords = f["PartType4"]["Coordinates"][:]
            star_mass = f["PartType4"]["Masses"][:]

            star_r = get_radii(star_coords)

        else:

            star_r = np.array([])
            star_mass = np.array([])

        if "PartType0" in f:

            gas_coords = f["PartType0"]["Coordinates"][:]
            gas_mass = f["PartType0"]["Masses"][:]

            gas_r = get_radii(gas_coords)

        else:

            gas_r = np.array([])
            gas_mass = np.array([])
    dm_order = np.argsort(dm_r)

    dm_r_sorted = dm_r[dm_order]
    dm_mass_sorted = dm_mass[dm_order]

    dm_cumulative = np.cumsum(dm_mass_sorted)
    star_order = np.argsort(star_r)

    star_r_sorted = star_r[star_order]
    star_mass_sorted = star_mass[star_order]

    star_cumulative = np.cumsum(star_mass_sorted)
    if len(gas_r) > 0:

        gas_order = np.argsort(gas_r)

        gas_r_sorted = gas_r[gas_order]
        gas_mass_sorted = gas_mass[gas_order]

        gas_cumulative = np.cumsum(gas_mass_sorted)

    else:

        gas_r_sorted = np.array([])
        gas_mass_sorted = np.array([])
        gas_cumulative = np.array([])

    r_aperture = 2.0 * Rh

    dm_mask = dm_r <= r_aperture
    star_mask = star_r <= r_aperture
    gas_mask = gas_r <= r_aperture

    particle_dm_2Rh = np.sum(dm_mass[dm_mask])
    particle_star_2Rh = np.sum(star_mass[star_mask])
    particle_gas_2Rh = np.sum(gas_mass[gas_mask])

    particle_total_2Rh = (
        particle_dm_2Rh
        + particle_star_2Rh
        + particle_gas_2Rh
    )

    print()
    print("MASS WITHIN 2 R_half")
    print("-" * 50)
    print("DM:")
    print(f"  particle = {particle_dm_2Rh:.6e}")
    print(f"  catalog  = {catalog_dm_2Rh:.6e}")

    print("Stars:")
    print(f"  particle = {particle_star_2Rh:.6e}")
    print("  catalog stellar mass is not currently stored")

    print("Gas:")
    print(f"  particle = {particle_gas_2Rh:.6e}")
    print()
    print("TOTAL:")
    print(f"  particle = {particle_total_2Rh:.6e}")
    print(f"  catalog  = {catalog_total_2Rh:.6e}")

    if catalog_dm_2Rh != 0:

        dm_difference = (
            particle_dm_2Rh - catalog_dm_2Rh
        ) / catalog_dm_2Rh

        print()
        print(
            f"DM fractional difference = "
            f"{dm_difference:+.3%}"
        )

    if catalog_total_2Rh != 0:

        total_difference = (
            particle_total_2Rh - catalog_total_2Rh
        ) / catalog_total_2Rh

        print(
            f"Total fractional difference = "
            f"{total_difference:+.3%}"
        )

    dm_r_kpc = dm_r_sorted / h
    star_r_kpc = star_r_sorted / h
    gas_r_kpc = gas_r_sorted / h

    max_length = max(
        len(dm_r_kpc),
        len(star_r_kpc),
        len(gas_r_kpc)
    )

    profile = pd.DataFrame({
        "SubhaloID": subhalo_id,

        "r_DM_kpc": pd.Series(
            np.pad(
                dm_r_kpc,
                (0, max_length - len(dm_r_kpc)),
                constant_values=np.nan
            )
        ),

        "M_DM_cumulative_1e10Msun_h": pd.Series(
            np.pad(
                dm_cumulative,
                (0, max_length - len(dm_cumulative)),
                constant_values=np.nan
            )
        ),

        "r_star_kpc": pd.Series(
            np.pad(
                star_r_kpc,
                (0, max_length - len(star_r_kpc)),
                constant_values=np.nan
            )
        ),

        "M_star_cumulative_1e10Msun_h": pd.Series(
            np.pad(
                star_cumulative,
                (0, max_length - len(star_cumulative)),
                constant_values=np.nan
            )
        ),
        "M_gas_cumulative_1e10Msun_h": pd.Series(
            np.pad(
                gas_cumulative,
                (0, max_length - len(gas_cumulative)),
                constant_values=np.nan
            )
        ),
    })

    # Add useful metadata
    profile["R_half_kpc"] = Rh / h
    profile["R_2half_kpc"] = 2 * Rh / h

    # Save individual profile
    output_file = (
        profile_dir /
        f"subhalo_{subhalo_id}_mass_profile.csv"
    )

    profile.to_csv(output_file, index=False)

    all_profiles.append(profile)

    print()
    print("Saved:")
    print(output_file)

print()
print("=" * 70)
print("ALL MASS PROFILES COMPLETE")
print("=" * 70)

print()
print(f"Profile files saved in:")
print(profile_dir)


MASS PROFILE: SUBHALO 738558

MASS WITHIN 2 R_half
--------------------------------------------------
DM:
  particle = 1.035691e-01
  catalog  = 1.035691e-01
Stars:
  particle = 1.685838e-01
  catalog stellar mass is not currently stored
Gas:
  particle = 0.000000e+00

TOTAL:
  particle = 2.721529e-01
  catalog  = 2.721529e-01

DM fractional difference = +0.000%
Total fractional difference = -0.000%

Saved:
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\particle_profiles_099\subhalo_738558_mass_profile.csv

MASS PROFILE: SUBHALO 355308

MASS WITHIN 2 R_half
--------------------------------------------------
DM:
  particle = 9.161883e-02
  catalog  = 9.161884e-02
Stars:
  particle = 1.530270e-01
  catalog stellar mass is not currently stored
Gas:
  particle = 0.000000e+00

TOTAL:
  particle = 2.446458e-01
  catalog  = 2.446458e-01

DM fractional difference = -0.000%
Total fractional difference = +0.000%

Saved:
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\dat

In [ ]:

import matplotlib.pyplot as plt

# Two representative galaxies
plot_ids = [
    738558,
    1436856,
    617680,
    1436856,
    340851
]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(11, 4.5),
    sharey=True
)
axes = axes.flatten()

for ax, subhalo_id in zip(axes, plot_ids):


    profile_file = (
        profile_dir /
        f"subhalo_{subhalo_id}_mass_profile.csv"
    )

    profile = pd.read_csv(profile_file)

    dm_mask = profile["r_DM_kpc"].notna()

    r_dm = profile.loc[
        dm_mask,
        "r_DM_kpc"
    ].to_numpy()

    m_dm = profile.loc[
        dm_mask,
        "M_DM_cumulative_1e10Msun_h"
    ].to_numpy()
    m_dm = m_dm * 1e10 / h

    star_mask = profile["r_star_kpc"].notna()

    r_star = profile.loc[
        star_mask,
        "r_star_kpc"
    ].to_numpy()

    m_star = profile.loc[
        star_mask,
        "M_star_cumulative_1e10Msun_h"
    ].to_numpy()

    m_star = m_star * 1e10 / h


    all_radii = np.unique(
        np.concatenate([
            r_dm,
            r_star,
        ])
    )
    total_dm = np.interp(
        all_radii,
        r_dm,
        m_dm,
        left=0,
        right=m_dm[-1]
    )

    total_star = np.interp(
        all_radii,
        r_star,
        m_star,
        left=0,
        right=m_star[-1]
    )



    total_gas = np.zeros_like(all_radii)

    total_mass = (
        total_dm
        + total_star
        + total_gas
    )

    row = catalog[
        catalog["SubhaloID"] == subhalo_id
    ].iloc[0]

    Rh = row["R_half_star"] / h

    ax.plot(
        r_dm,
        m_dm,
        color="black",
        linewidth=2,
        label="DM"
    )

    ax.plot(
        r_star,
        m_star,
        color="red",
        linewidth=2,
        label="Star"
    )

    ax.plot(
        all_radii,
        total_mass,
        color="gray",
        linewidth=2,
        label="Total"
    )

    ax.axvline(
        Rh,
        color="green",
        linestyle=":",
        linewidth=2
    )

    ax.set_xscale("linear")

    ax.set_yscale("log")

    ax.set_xlim(0, 5)

    ax.set_ylim(
        1e6,
        1e10
    )

    ax.set_xlabel(r"$r$ [kpc]")

    ax.set_title(
        f"Subhalo {subhalo_id}"
    )

    ax.grid(
        True,
        which="both",
        alpha=0.2
    )
axes[0].set_ylabel(
    r"$M(<r)$ [$M_\odot$]"
)
axes[0].legend(
    loc="upper left",
    frameon=False
)

plt.tight_layout()

output_file = (
    project_dir
    / "figures"
    / "mass_profiles_two_panel.png"
)

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
for ax in axes[len(plot_ids):]:
    ax.set_visible(False)
print()
print("Saved:")
print(output_file)